# 07b · Nulo de estructura constante para la parte dinámica (iteración 3b, tarea D7)

**Objetivo.** Primera pieza de OE11 (y de HT1): una distribución de referencia para los $k^*$ de
`aede` y para el coeficiente RV entre ventanas, bajo la hipótesis de que la estructura de
correlaciones **no cambia** en el tiempo. Sin este nulo, la grilla de anchos, pesos, maxcl y
criterios (notebook 06c) sería un «jardín de senderos» (plan de la iteración 3b, §1 y riesgo 1).
Toda lectura de un $k^*$ o de un corte se hace contra este nulo, nunca contra los episodios.

**Construcción** (remuestreo por bloques **conjunto**, $L = 24$; sensibilidad con $L = 12$ y
$L = 36$, riesgo 2 del plan): los mismos inicios de bloque para las 17 series.

- Vía transformada: se remuestrean las filas.
- Vía sin transformar: se remuestrean los incrementos mensuales y se acumulan desde 0, como la
  versión «con deriva» del nulo de tendencias de la sección 2 del Rmd.

Así se conservan la correlación contemporánea y la dinámica corta, pero se dispersa en el tiempo
cualquier cambio de estructura. *Supuesto declarado:* el bloque conjunto mezcla tramos de
regímenes distintos; si hay un cambio fuerte de varianza (2020), el nulo puede ser «demasiado
fácil» de superar (riesgo 2).

**Medidas** (`src/nulo_estructura.py`), para cada réplica, ancho $n_w$ ∈ {24, 36, 48, 60} y
esquema de pesos (uniformes, gaussianos `nsd = 3`):

1. el $k^*$ de cada criterio de `aede` con maxcl = 20 (codo por caída máxima, CH con $k$ y con
   $k-1$ grados de libertad);
2. el perfil del coeficiente RV entre **ventanas adyacentes no solapadas** (D10): RV entre las
   cargas de `aede` (`correlationlsq`, 17 series × 3 dimensiones, d = 3 por D9) de la ventana que
   empieza en la fila $i$ y de la que empieza en $i + n_w$.

**Réplicas y paridad (D6).** El guion `R/07b_nulo_estructura.R` sortea 200 réplicas por longitud
de bloque (semilla 2026) y escribe los inicios en `R_nulo_estructura_indices_L{12,24,36}.csv`; este
notebook usa esas mismas réplicas (`SEMILLA = None`: no sortea). R calcula las **20 primeras**
réplicas de cada $L$ y la paridad completa se exige en ellas; las réplicas 21 a 200 con $L = 24$ se
calculan **solo en Python** (columna `en_paridad`), como declara la tabla de paridad del Rmd.
La réplica 0 es la serie observada.

**Salidas** (niveles en `data/results/niveles/`, transformadas en `data/results/`):
`07b_nulo_estructura_kstar.csv`, `07b_nulo_estructura_kstar_posicion.csv`,
`07b_nulo_estructura_rv.csv` (perfiles completos, $L = 24$, réplicas 0 a 20),
`07b_nulo_estructura_rv_resumen.csv` (resumen por réplica) y `07b_nulo_estructura_rv_nulo.csv`
(cuantiles del RV nulo agrupado por ancho y pesos).

In [1]:
# ---- Configuración --------------------------------------------------------------
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src import nulo_estructura as ne
from src.datos import cargar_vias
from src.remuestreo import leer_inicios_bootstrap

SEMILLA = None
REPLICAS = {12: 20, 24: 200, 36: 20}      # réplicas por longitud de bloque
B_PARIDAD = 20
DIR_T = RAIZ / 'data' / 'results'
DIR_N = DIR_T / 'niveles'
X, Xt = cargar_vias(RAIZ)
VIAS = {'niveles': (X, DIR_N), 'transformadas': (Xt, DIR_T)}
INICIOS = {L: leer_inicios_bootstrap(DIR_T / f'R_nulo_estructura_indices_L{L}.csv') for L in REPLICAS}
for L, ini in INICIOS.items():
    # Los índices son los mismos en las dos carpetas (las dos vías remuestrean 227 filas)
    assert (ini == leer_inicios_bootstrap(DIR_N / f'R_nulo_estructura_indices_L{L}.csv')).all()
    assert ini.shape[0] >= REPLICAS[L] and ini.min() >= 1 and ini.max() <= 227 - L + 1
    print(f'L = {L}: {ini.shape[0]} réplicas sorteadas, {ini.shape[1]} bloques; se usan {REPLICAS[L]}')

L = 12: 200 réplicas sorteadas, 19 bloques; se usan 20
L = 24: 200 réplicas sorteadas, 10 bloques; se usan 200
L = 36: 200 réplicas sorteadas, 7 bloques; se usan 20


## 1. Serie observada y réplicas nulas

In [2]:
resultados = {}
t0 = time.time()
for via, (M, d) in VIAS.items():
    k0, rv0, rr0 = ne.medir(M)
    kst = [k0.assign(L=0, replica=0)]
    rvp = [rv0.assign(replica=0)]
    rvr = [rr0.assign(L=0, replica=0)]
    rv_todos = {}                                     # (L, nw, pesos) -> lista de RV de todas las réplicas
    for L, B in REPLICAS.items():
        for b in range(1, B + 1):
            k, rv, rr = ne.medir(ne.serie_nula(M, via, INICIOS[L][b - 1], L))
            kst.append(k.assign(L=L, replica=b))
            rvr.append(rr.assign(L=L, replica=b))
            if L == 24 and b <= B_PARIDAD:
                rvp.append(rv.assign(replica=b))
            for (nw, pz), g in rv.groupby(['nw', 'pesos']):
                rv_todos.setdefault((L, nw, pz), []).append(g.rv.to_numpy())
        print(f'{via}: L = {L}, {B} réplicas ({(time.time() - t0) / 60:.1f} min)')
    resultados[via] = {'kstar': pd.concat(kst, ignore_index=True), 'rv': pd.concat(rvp, ignore_index=True),
                       'rv_resumen': pd.concat(rvr, ignore_index=True), 'rv_todos': rv_todos}

niveles: L = 12, 20 réplicas (0.6 min)


niveles: L = 24, 200 réplicas (5.9 min)


niveles: L = 36, 20 réplicas (6.5 min)


transformadas: L = 12, 20 réplicas (7.1 min)


transformadas: L = 24, 200 réplicas (12.3 min)


transformadas: L = 36, 20 réplicas (12.8 min)


## 2. Tablas de salida y posición de cada $k^*$ observado en el nulo

`prop_nulo_ge_obs` es la proporción de réplicas con $k^*$ mayor o igual que el observado y
`prop_nulo_le_obs`, con $k^*$ menor o igual; `prop_nulo_en_tope`, la proporción de réplicas cuyo
criterio elige el tope (k = 20). El $k^*$ observado con maxcl = 20 es el de `06c_aede_cortes.csv`.

In [3]:
for via, (M, d) in VIAS.items():
    r = resultados[via]
    ks = r['kstar'][['L', 'replica', 'nw', 'pesos', 'criterio', 'k', 'k_en_tope']].copy()
    ks['en_paridad'] = ks.replica <= B_PARIDAD
    ks.to_csv(d / '07b_nulo_estructura_kstar.csv', index=False)
    r['rv'][['replica', 'nw', 'pesos', 'fecha_a', 'fecha_b', 'rv']].to_csv(d / '07b_nulo_estructura_rv.csv', index=False)
    rr = r['rv_resumen'][['L', 'replica', 'nw', 'pesos', 'n_pares', 'rv_min', 'rv_p05', 'rv_p50', 'rv_media']].copy()
    rr['en_paridad'] = rr.replica <= B_PARIDAD
    rr.to_csv(d / '07b_nulo_estructura_rv_resumen.csv', index=False)
    # Cuantiles del RV nulo agrupado (todas las posiciones de todas las réplicas de cada L)
    filas = []
    for (L, nw, pz), lst in sorted(r['rv_todos'].items()):
        v = np.concatenate(lst)
        filas.append({'L': L, 'nw': nw, 'pesos': pz, 'replicas': len(lst), 'n_valores': len(v),
                      'p01': np.quantile(v, 0.01), 'p05': np.quantile(v, 0.05), 'p10': np.quantile(v, 0.10),
                      'p50': np.quantile(v, 0.50)})
    pd.DataFrame(filas).to_csv(d / '07b_nulo_estructura_rv_nulo.csv', index=False)
    # Posición del k* observado (réplica 0) en la distribución nula de cada L
    obs = ks[ks.replica == 0].set_index(['nw', 'pesos', 'criterio']).k
    g06c = pd.read_csv(d / '06c_aede_cortes.csv')
    g06c = g06c[g06c.maxcl == 20].set_index(['nw', 'pesos', 'criterio']).k
    assert (obs.sort_index() == g06c.sort_index()).all(), 'el k* observado no coincide con 06c (maxcl = 20)'
    pos = []
    for L in REPLICAS:
        nul = ks[(ks.L == L) & (ks.replica > 0)]
        for (nw, pz, cr), g in nul.groupby(['nw', 'pesos', 'criterio'], sort=False):
            k_obs = int(obs.loc[(nw, pz, cr)])
            pos.append({'L': L, 'nw': nw, 'pesos': pz, 'criterio': cr, 'k_obs': k_obs, 'k_obs_en_tope': k_obs == ne.MAXCL,
                        'replicas': len(g), 'nulo_p05': np.quantile(g.k, 0.05), 'nulo_p50': np.quantile(g.k, 0.5),
                        'nulo_p95': np.quantile(g.k, 0.95), 'prop_nulo_ge_obs': float(np.mean(g.k >= k_obs)),
                        'prop_nulo_le_obs': float(np.mean(g.k <= k_obs)), 'prop_nulo_en_tope': float(np.mean(g.k_en_tope))})
    pos = pd.DataFrame(pos)
    pos.to_csv(d / '07b_nulo_estructura_kstar_posicion.csv', index=False)
    print(via)
    display(pos[pos.L == 24])

niveles


,L,nw,pesos,criterio,k_obs,k_obs_en_tope,replicas,nulo_p05,nulo_p50,nulo_p95,prop_nulo_ge_obs,prop_nulo_le_obs,prop_nulo_en_tope
24,24,24,uniformes,codo (caída máxima),6,False,200,2.00,4.0,9.00,0.240,0.885,0.000
25,24,24,uniformes,Calinski-Harabász k gl (aede),20,True,200,17.90,20.0,20.00,0.905,1.000,0.905
26,24,24,uniformes,Calinski-Harabász k-1 gl (habitual),20,True,200,2.00,20.0,20.00,0.710,1.000,0.710
27,24,24,gaussianos_nsd3,codo (caída máxima),6,False,200,2.00,3.5,9.05,0.270,0.825,0.000
28,24,24,gaussianos_nsd3,Calinski-Harabász k gl (aede),20,True,200,15.95,20.0,20.00,0.900,1.000,0.900
29,24,24,gaussianos_nsd3,Calinski-Harabász k-1 gl (habitual),20,True,200,2.00,20.0,20.00,0.740,1.000,0.740
30,24,36,uniformes,codo (caída máxima),2,False,200,2.00,3.0,8.00,1.000,0.290,0.000
31,24,36,uniformes,Calinski-Harabász k gl (aede),20,True,200,20.00,20.0,20.00,1.000,1.000,1.000
32,24,36,uniformes,Calinski-Harabász k-1 gl (habitual),2,False,200,2.00,20.0,20.00,1.000,0.060,0.910
33,24,36,gaussianos_nsd3,codo (caída máxima),2,False,200,2.00,4.0,8.00,1.000,0.270,0.000


transformadas


,L,nw,pesos,criterio,k_obs,k_obs_en_tope,replicas,nulo_p05,nulo_p50,nulo_p95,prop_nulo_ge_obs,prop_nulo_le_obs,prop_nulo_en_tope
24,24,24,uniformes,codo (caída máxima),4,False,200,2.00,4.0,7.00,0.555,0.665,0.000
25,24,24,uniformes,Calinski-Harabász k gl (aede),20,True,200,6.95,20.0,20.00,0.800,1.000,0.800
26,24,24,uniformes,Calinski-Harabász k-1 gl (habitual),20,True,200,2.00,20.0,20.00,0.590,1.000,0.590
27,24,24,gaussianos_nsd3,codo (caída máxima),10,False,200,2.00,4.0,9.05,0.050,0.960,0.000
28,24,24,gaussianos_nsd3,Calinski-Harabász k gl (aede),20,True,200,19.00,20.0,20.00,0.905,1.000,0.905
29,24,24,gaussianos_nsd3,Calinski-Harabász k-1 gl (habitual),20,True,200,3.00,20.0,20.00,0.755,1.000,0.755
30,24,36,uniformes,codo (caída máxima),3,False,200,2.00,4.0,7.00,0.740,0.440,0.000
31,24,36,uniformes,Calinski-Harabász k gl (aede),20,True,200,16.90,20.0,20.00,0.905,1.000,0.905
32,24,36,uniformes,Calinski-Harabász k-1 gl (habitual),20,True,200,2.00,20.0,20.00,0.775,1.000,0.775
33,24,36,gaussianos_nsd3,codo (caída máxima),3,False,200,2.00,4.0,7.00,0.775,0.475,0.000


## 3. Comparación con R (20 réplicas por longitud de bloque y serie observada)

$k^*$ exactos; perfiles RV ($L = 24$) y resúmenes por réplica con tolerancia $10^{-10}$.

In [4]:
for via, (M, d) in VIAS.items():
    f_k = d / 'R_nulo_estructura_kstar.csv'
    if not f_k.exists():
        print(f'{via}: aún no existe {f_k.name} (ejecutar R/07b_nulo_estructura.R)')
        continue
    rk = pd.read_csv(f_k)
    pk = pd.read_csv(d / '07b_nulo_estructura_kstar.csv')
    clave = ['L', 'replica', 'nw', 'pesos', 'criterio']
    m = pk.merge(rk, on=clave, suffixes=('', '_r'))
    n_r = rk.groupby('L').replica.max().to_dict()
    assert len(m) == len(rk), (via, len(m), len(rk))
    n_dif = int((m.k != m.k_r).sum())
    pr = pd.read_csv(d / '07b_nulo_estructura_rv.csv').merge(pd.read_csv(d / 'R_nulo_estructura_rv.csv'),
                                                             on=['replica', 'nw', 'pesos', 'fecha_a', 'fecha_b'], suffixes=('', '_r'))
    dif_rv = float(np.max(np.abs(pr.rv - pr.rv_r)))
    rs = pd.read_csv(d / '07b_nulo_estructura_rv_resumen.csv').merge(pd.read_csv(d / 'R_nulo_estructura_rv_resumen.csv'),
                                                                     on=['L', 'replica', 'nw', 'pesos'], suffixes=('', '_r'))
    dif_rs = max(float(np.max(np.abs(rs[c] - rs[c + '_r']))) for c in ['rv_min', 'rv_p05', 'rv_p50', 'rv_media'])
    print(f'{via}: réplicas en R por L {n_r}; k* distintos {n_dif} de {len(m)}; '
          f'dif. máx. RV {dif_rv:.1e} ({len(pr)} pares); resúmenes {dif_rs:.1e}')
    assert n_dif == 0 and dif_rv < 1e-10 and dif_rs < 1e-10, via

niveles: réplicas en R por L {0: 0, 12: 20, 24: 20, 36: 20}; k* distintos 0 de 1464; dif. máx. RV 8.3e-15 (24360 pares); resúmenes 1.9e-15
transformadas: réplicas en R por L {0: 0, 12: 20, 24: 20, 36: 20}; k* distintos 0 de 1464; dif. máx. RV 3.3e-14 (24192 pares); resúmenes 7.4e-15


**Lectura.** Sin interpretación en esta fase: la posición de cada $k^*$ observado en el nulo y la
proporción de pares adyacentes con RV bajo el p5 nulo (criterio de OE4, D10) se leen en la fase 1
(tareas D10 y D11 del plan), junto con la marca de borde de ventana (D7, D13).